In [1]:
import os
import zipfile

# 0. Setup Kaggle Credentials (local)
# Make sure you have your kaggle.json in C:\Users\<YourUser>\.kaggle\kaggle.json
# OR set environment variables directly:
os.environ['KAGGLE_USERNAME'] = "anmolthakur70"
os.environ['KAGGLE_KEY'] = "KGAT_661b4277d49081710cce606a4e8caeda"

# Define local data path
dataset_dir = os.path.join(os.getcwd(), "..", "data")
if not os.path.exists(dataset_dir):
    os.makedirs(dataset_dir)

dataset_name = "abdallahalidev/plantvillage-dataset"
zip_path = os.path.join(dataset_dir, "plantvillage-dataset.zip")
extract_path = os.path.join(dataset_dir, "plantvillage_data")

# Download and Unzip if not exists
if not os.path.exists(extract_path):
    print("Downloading dataset...")
    # Using the kaggle CLI command (requires kaggle installed: pip install kaggle)
    # The ! prefix runs terminal commands in Jupyter
    !kaggle datasets download -d abdallahalidev/plantvillage-dataset -p "{dataset_dir}" --force
    
    print("Unzipping...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Download and unzip complete!")
else:
    print(f"Data is already present at {extract_path}")

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0

Unzipping...



  0%|          | 0.00/2.04G [00:00<?, ?B/s]
  5%|▍         | 97.0M/2.04G [00:00<00:02, 1.01GB/s]
  9%|▉         | 193M/2.04G [00:00<00:02, 877MB/s]  
 13%|█▎        | 278M/2.04G [00:00<00:02, 825MB/s]
 17%|█▋        | 359M/2.04G [00:00<00:02, 833MB/s]
 21%|██        | 442M/2.04G [00:00<00:02, 845MB/s]
 25%|██▌       | 523M/2.04G [00:00<00:01, 828MB/s]
 29%|██▉       | 603M/2.04G [00:00<00:01, 793MB/s]
 33%|███▎      | 679M/2.04G [00:00<00:01, 753MB/s]
 36%|███▋      | 757M/2.04G [00:00<00:01, 750MB/s]
 40%|███▉      | 833M/2.04G [00:01<00:01, 763MB/s]
 43%|████▎     | 907M/2.04G [00:01<00:01, 762MB/s]
 47%|████▋     | 983M/2.04G [00:01<00:01, 752MB/s]
 51%|█████     | 1.04G/2.04G [00:01<00:01, 768MB/s]
 54%|█████▍    | 1.11G/2.04G [00:01<00:01, 756MB/s]
 58%|█████▊    | 1.18G/2.04G [00:01<00:01, 754MB/s]
 62%|██████▏   | 1.26G/2.04G [00:01<00:01, 775MB/s]
 65%|██████▌   | 1.33G/2.04G [00:01<00:01, 758MB/s]
 69%|██████▊   | 1.40G/2.04G [00:01<00:00, 757MB/s]
 73%|███████▎  | 1.48G/2.04

Download and unzip complete!


In [2]:
!pip install tensorflow -q

In [ ]:
import tensorflow as tf
import os

# 1. Setup paths and parameters
# Adjust this path based on the actual extracted structure. 
# Usually it extracts to a subfolder like 'plantvillage dataset/color'
dataset_root = os.path.join(os.getcwd(), "..", "data", "plantvillage_data")
# Check if there is a nested folder structure
possible_path = os.path.join(dataset_root, "plantvillage dataset", "color")
if os.path.exists(possible_path):
    DATA_DIR = possible_path
else:
    DATA_DIR = dataset_root # Fallback

print(f"Loading data from: {DATA_DIR}")

BATCH_SIZE = 32
IMG_SIZE = (224, 224)
EPOCHS = 5 # Reduced for quicker local testing, increase for better accuracy

print("1. Loading datasets...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, shuffle=True, validation_split=0.2, subset="training",
    seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, shuffle=True, validation_split=0.2, subset="validation",
    seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

NUM_CLASSES = len(train_dataset.class_names)
print(f"--> Found {NUM_CLASSES} classes: {train_dataset.class_names}")

# Optimize for speed
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# 2. Build the Model Architecture
print("\n2. Building the Neural Network...")
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal_and_vertical'),
  tf.keras.layers.RandomRotation(0.2),
])

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet'
)
base_model.trainable = False # Freeze base weights

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# 3. Train the Model
print(f"\n3. Starting Training for {EPOCHS} epochs...")
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

# 4. Save to Backend Models Folder
model_dir = os.path.join(os.getcwd(), "..", "backend", "models")
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

model_path = os.path.join(model_dir, "crop_sentinel_mobilenetv2.keras")
model.save(model_path)
print(f"\n✅ Model successfully saved to {model_path}!")

Loading data from: c:\crop-disease-forecast\notebooks\..\data\plantvillage_data\plantvillage dataset\color
1. Loading datasets...
Found 54305 files belonging to 38 classes.
Using 43444 files for training.
Found 54305 files belonging to 38 classes.
Using 10861 files for validation.
--> Found 38 classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___hea

: 